In [1]:
import numpy as np
import numpy_financial as npf
import pandas as pd

# --- Project & Financial Inputs ---
CAPEX = 50_000_000         # Initial Capital Expenditure (€50M)
PROJECT_LIFETIME = 15      # Years
DISCOUNT_RATE = 0.08       # Weighted Average Cost of Capital (WACC/Discount Rate)
DAILY_CYCLES_TARGET = 1.5  # Strategic goal: 1.5 equivalent full cycles per day
MAX_CYCLES_SPEC = 5000     # Warranty limit (determines degradation sensitivity)
ANNUAL_OM_COST = 500_000   # Annual Operating & Maintenance cost (€500k/yr)


In [3]:
# --- 1. Strategy Model (Inputs from Operational/Degradation decisions) ---

def model_annual_revenue(daily_profit_base, target_cycles, capacity_mwh, start_year=1):
    """
    Simulates the declining annual revenue based on capacity degradation.
    Assumption: Revenue is proportional to available capacity (SoH) and installed MWh.
    Degradation: Simplified linear loss based on cumulative cycles (a key strategic risk).
    """

    annual_revenues = {}
    current_soh = 1.0  # Starts at 100%
    DAYS_PER_YEAR = 365

    # Revenue scales with installed capacity (MWh) so changing capacity_mwh matters
    capacity_scale = capacity_mwh / 100  # Normalised to 100 MWh baseline

    # Calculate lifetime cycles consumed by strategy
    annual_cycles_consumed = target_cycles * DAYS_PER_YEAR

    for year in range(start_year, PROJECT_LIFETIME + 1):

        # 1. Capacity Loss Model (Degradation)
        # Use a simplified linear degradation based on total cycles vs warranty limit
        # This penalizes high cycling (high DoD)
        soh_loss_rate = annual_cycles_consumed / MAX_CYCLES_SPEC
        current_soh = np.maximum(0.80, current_soh - soh_loss_rate)  # Floor at 80%

        # 2. Revenue Calculation (scales with capacity and SoH)
        annual_profit = daily_profit_base * DAYS_PER_YEAR * current_soh * capacity_scale
        annual_revenues[year] = annual_profit

    return annual_revenues, current_soh


In [4]:
# --- 2. Valuation Model (Discounted Cash Flow - DCF) ---

def calculate_dcf_metrics(annual_cash_flows, initial_capex, discount_rate, annual_om_cost=0):
    """Calculates Net Present Value (NPV) and Internal Rate of Return (IRR).

    Args:
        annual_cash_flows: dict mapping year -> gross revenue
        initial_capex: upfront investment (negative cash flow at year 0)
        discount_rate: WACC used to discount future cash flows
        annual_om_cost: annual O&M expenses subtracted from each year's gross revenue
    """

    # NPV Calculation
    npv = -initial_capex
    net_cash_flows = []
    for year, cash_flow in annual_cash_flows.items():
        net_cf = cash_flow - annual_om_cost
        discount_factor = 1 / (1 + discount_rate) ** year
        npv += net_cf * discount_factor
        net_cash_flows.append(net_cf)

    # IRR Calculation using numpy_financial (np.irr was removed in NumPy 1.20)
    cf_list = [-initial_capex] + net_cash_flows
    try:
        irr = npf.irr(cf_list)
    except Exception as e:
        print(f"Warning: IRR calculation failed: {e}")
        irr = np.nan

    return npv, irr


In [5]:
# 1. Sample Strategic Input (e.g., from Stochastic Optimization)
# Assume the optimal *risk-averse* dispatch yields a gross profit of
# €80,000 per day IF the battery were 100% efficient and new.
BASE_DAILY_PROFIT_NEW = 80_000
BESS_MWh = 100  # Assuming 100 MWh BESS

# 2. Run Strategy Model (Generate Revenue Stream based on degradation)
annual_revenues, final_soh = model_annual_revenue(
    BASE_DAILY_PROFIT_NEW, DAILY_CYCLES_TARGET, BESS_MWh
)

# 3. Run Valuation Model (DCF) — now includes O&M costs
npv, irr = calculate_dcf_metrics(
    annual_revenues, CAPEX, DISCOUNT_RATE, annual_om_cost=ANNUAL_OM_COST
)


In [ ]:
# 4. Results Reporting
print("--- Asset Strategy & Valuation Results ---")
print(f"BESS Capacity: {BESS_MWh} MWh | Strategic Cycles/Day: {DAILY_CYCLES_TARGET}")
print(f"Project Lifetime: {PROJECT_LIFETIME} years | Discount Rate: {DISCOUNT_RATE*100:.1f}%")
print(f"Annual O&M Cost: {ANNUAL_OM_COST:,.0f} €")
print("-" * 50)
print(f"Final State of Health (SoH): {final_soh*100:.1f}%")
print(f"Initial Investment (CAPEX): {CAPEX:,.0f} €")
print(f"Calculated Net Present Value (NPV): {npv:,.0f} €")
if not np.isnan(irr):
    print(f"Calculated Internal Rate of Return (IRR): {irr*100:.2f}%")
else:
    print("Calculated Internal Rate of Return (IRR): N/A (calculation failed)")
print("-" * 50)

# Strategic Conclusion
if npv > 0:
    print(f"
STRATEGY CONCLUSION: The NPV is positive, meaning the asset strategy is financially sound. "
          f"The project is expected to generate {npv:,.0f} € more than the cost of capital, justifying the investment.")
else:
    print("
STRATEGY CONCLUSION: The NPV is negative. The current operational strategy (or high initial cost) "
          "makes the project unprofitable over its lifetime.")
